# 🎮 PocketPal - Pally AI Training with Unsloth

This notebook fine-tunes a small LLM (Llama 3.1 8B, Mistral 7B, or others) to become **Pally**, PocketPal's friendly AI money coach for Indian students.

## Features Pally will learn:
- Friendly, supportive personality with Hindi/Hinglish flair
- Tool calling for wallet, spending, goals, streaks, badges, etc.
- Playful roasting without shaming
- Financial awareness guidance

## 1️⃣ Install Dependencies

In [ ]:
%%capture
!pip install unsloth
# Get latest Unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2️⃣ Upload Your Training Data

Upload your `train_sharegpt.jsonl`, `valid_sharegpt.jsonl`, and `test_sharegpt.jsonl` files.

In [ ]:
from google.colab import files
import os

# Create data directory
os.makedirs('data', exist_ok=True)

print("📁 Upload your training data files:")
print("   - train_sharegpt.jsonl")
print("   - valid_sharegpt.jsonl (optional)")
print("   - test_sharegpt.jsonl (optional)")
print()

uploaded = files.upload()

# Move files to data directory
for filename in uploaded.keys():
    os.rename(filename, f'data/{filename}')
    print(f"✅ Moved {filename} to data/")

## 3️⃣ Load the Base Model

We recommend **Llama 3.1 8B Instruct** as it is the current state-of-the-art for this size class, beating Mistral 7B v0.3 in logic and reasoning. 

However, you can also stick with **Mistral 7B v0.3** if you prefer its behavior.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048  # Adjust based on your data
dtype = None  # Auto-detect (Float16 for Tesla T4, BFloat16 for Ampere+)
load_in_4bit = True  # Use 4-bit quantization for memory efficiency

# Choose your base model
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"  # SOTA 8B model, recommended upgrade
# model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"  # The model you used before
# model_name = "unsloth/Llama-3.2-3B-Instruct"  # Faster & cheaper, good quality
# model_name = "unsloth/Qwen2.5-7B-Instruct"  # Also excellent, very strong logic

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"✅ Loaded {model_name}")

## 4️⃣ Add LoRA Adapters

LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training a small subset of parameters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank (higher = more capacity, more memory)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",  # MLP layers
    ],
    lora_alpha=16,
    lora_dropout=0,  # 0 is optimized
    bias="none",  # "none" is optimized
    use_gradient_checkpointing="unsloth",  # Long context support
    random_state=3407,
    use_rslora=False,  # Rank-stabilized LoRA
    loftq_config=None,
)

print("✅ LoRA adapters added")
print(f"   Trainable parameters: {model.print_trainable_parameters()}")

## 5️⃣ Load and Prepare Dataset

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Apply chat template for the model
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",  # Use llama-3.1 template for Llama models
    # chat_template="mistral",  # Use for Mistral models if not auto-detected
)

# Load datasets
dataset = load_dataset(
    "json",
    data_files={
        "train": "data/train_sharegpt.jsonl",
        "validation": "data/valid_sharegpt.jsonl",
    }
)

print(f"✅ Loaded dataset:")
print(f"   Train: {len(dataset['train'])} examples")
print(f"   Validation: {len(dataset['validation'])} examples")

In [ ]:
# Format function to convert ShareGPT to model format
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = []
    
    for convo in convos:
        # Convert ShareGPT format to standard messages format
        messages = []
        for turn in convo:
            role = "user" if turn["from"] == "human" else "assistant"
            messages.append({"role": role, "content": turn["value"]})
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

# Preview a sample
print("📝 Sample formatted input:")
print(dataset["train"][0]["text"][:500] + "...")

## 6️⃣ Training Configuration

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can be True for short sequences
    args=TrainingArguments(
        # Training settings
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8
        
        # Learning rate
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        
        # Training duration
        num_train_epochs=3,  # Adjust based on dataset size
        # max_steps=100,  # Alternative: fixed number of steps
        
        # Precision
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        
        # Logging & Saving
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        
        # Output
        output_dir="outputs",
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=3407,
        report_to="none",  # Set to "wandb" for Weights & Biases logging
    ),
)

print("✅ Trainer configured")

## 7️⃣ Train! 🚀

In [ ]:
# Check GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"🖥️ GPU: {gpu_stats.name}")
print(f"   Memory: {start_gpu_memory} GB / {max_memory} GB")
print()
print("🎯 Starting training...")
print()

In [ ]:
# Train!
trainer_stats = trainer.train()

# Show final stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n✅ Training complete!")
print(f"   Peak GPU memory: {used_memory} GB")
print(f"   Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

## 8️⃣ Test the Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def chat_with_pally(user_message):
    """Chat with the trained Pally model."""
    messages = [
        {"role": "user", "content": user_message}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant response
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

# Test conversations
test_prompts = [
    "What's my balance?",
    "I spent too much this week",
    "Show me my spending breakdown",
    "Roast my spending habits",
    "Yaar budget kharab ho gaya",
]

print("🧪 Testing Pally...\n")
for prompt in test_prompts:
    print(f"👤 User: {prompt}")
    response = chat_with_pally(prompt)
    print(f"🤖 Pally: {response}")
    print("-" * 50)

## 9️⃣ Save the Model

Choose your preferred export format:

In [ ]:
# Option A: Save LoRA adapters only (small, requires base model)
model.save_pretrained("pally-lora")
tokenizer.save_pretrained("pally-lora")
print("✅ Saved LoRA adapters to pally-lora/")

In [ ]:
# Option B: Save merged model (16-bit, larger but standalone)
model.save_pretrained_merged("pally-merged", tokenizer, save_method="merged_16bit")
print("✅ Saved merged 16-bit model to pally-merged/")

In [ ]:
# Option C: Export to GGUF for llama.cpp / Ollama
# Choose quantization: q4_k_m (balanced), q5_k_m (better quality), q8_0 (best quality)
model.save_pretrained_gguf("pally-gguf", tokenizer, quantization_method="q4_k_m")
print("✅ Saved GGUF model to pally-gguf/")

## 🔟 Download the Model

In [ ]:
# Zip and download
import shutil

# Choose which to download
folder_to_download = "pally-lora"  # or "pally-merged" or "pally-gguf"

shutil.make_archive(folder_to_download, 'zip', folder_to_download)
files.download(f"{folder_to_download}.zip")
print(f"📥 Downloading {folder_to_download}.zip")

## 📤 Push to Hugging Face (Optional)

In [ ]:
# Login to Hugging Face
from huggingface_hub import login
login()  # Enter your token when prompted

In [ ]:
# Push to Hub
HF_USERNAME = "your-username"  # Change this!

# Push LoRA
model.push_to_hub(f"{HF_USERNAME}/pally-lora", token=True)
tokenizer.push_to_hub(f"{HF_USERNAME}/pally-lora", token=True)

# Or push merged model
# model.push_to_hub_merged(f"{HF_USERNAME}/pally-merged", tokenizer, save_method="merged_16bit", token=True)

# Or push GGUF
# model.push_to_hub_gguf(f"{HF_USERNAME}/pally-gguf", tokenizer, quantization_method="q4_k_m", token=True)

print(f"✅ Pushed to https://huggingface.co/{HF_USERNAME}/pally-lora")

---

## 🎉 Done!

Your Pally AI is now trained and ready. Next steps:

1. **For Ollama**: Use the GGUF export with `ollama create pally -f Modelfile`
2. **For Vercel AI SDK**: Use the merged model with Hugging Face Inference API
3. **For local testing**: Load the LoRA adapters with the base model